# Fine-tuning FLAN-T5-Large para Text-to-SQL

Este notebook entrena un modelo FLAN-T5-Large especializado en generar consultas SQL a partir de preguntas en lenguaje natural usando el dataset `gretelai/synthetic_text_to_sql`.

## ¿Por qué FLAN-T5-Large?
- **770M parámetros**: Comparable con Llama 3B en capacidad
- **Instrucciones pre-entrenadas**: Mejor comprensión de tareas
- **Arquitectura encoder-decoder**: Ideal para texto → SQL
- **Optimizado para seguir instrucciones**: Mayor precisión en SQL

## Dataset: gretelai/synthetic_text_to_sql
- **sql_context**: Schema de la base de datos
- **sql_prompt**: Pregunta en lenguaje natural
- **sql**: Query SQL esperada como respuesta
- **Filtros de calidad**: Elimina ejemplos con errores o muy cortos

## Configuración optimizada para Tesla T4
- **Memoria optimizada**: 8-bit quantization + device_map
- **LoRA fine-tuning**: Eficiente y rápido
- **Batch size adaptativo**: Evita OutOfMemory
- **Configuración profesional**: 770M parámetros vs 220M de T5-base

---

## 1. Autenticación en Hugging Face

Primero nos autenticamos para acceder a los modelos

In [ ]:
# Autenticación en Hugging Face
from huggingface_hub import login

# Ingresar token de Hugging Face
print("🔑 Autenticándose en Hugging Face...")

# Descomentar la siguiente línea y usar tu token
# login(token="tu_token_aqui")

# O usar login interactivo
login()

print("✅ Autenticación completada")

## 2. Imports y Dependencias

Importamos todas las librerías necesarias

In [ ]:
# ============================================
# IMPORTS COMPLETOS Y UNIFICADOS
# ============================================

# Framework principal de deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Transformers y Hugging Face
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# PEFT para LoRA
from peft import (
    get_peft_model,
    LoraConfig,
    PeftModel,
    TaskType
)

# Datasets
from datasets import Dataset as HFDataset, load_dataset

# Utilidades estándar
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
import logging
import warnings
import gc

# Configuración de warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.WARNING)

print("✅ Todos los imports cargados correctamente")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers disponible")
print(f"🔧 PEFT para LoRA disponible")
print(f"🎮 CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎯 GPUs detectadas: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1024**3:.1f}GB)")
else:
    print("⚠️ CUDA no disponible - usando CPU")

## 3. Configuraciones Principales

Definimos todas las configuraciones del modelo, entrenamiento y memoria

In [ ]:
# ============================================
# CONFIGURACIONES PRINCIPALES
# ============================================

    # Configuración del modelo
CONFIG = {
    # Modelo base
    "model_name": "google/flan-t5-large",       # 770M parámetros - comparable a Llama 3B
    
    # Configuración del dataset
    "dataset_name": "gretelai/synthetic_text_to_sql",  # Dataset de text-to-SQL
    "dataset_split": "train",                   # Split del dataset
    "max_samples": 1000,                        # REDUCIDO para Kaggle - 1000 muestras
    
    # Configuración de tokens
    "max_input_length": 512,                    # Longitud máxima de entrada
    "max_target_length": 128,                   # Longitud máxima de salida SQL
    
    # Configuración de entrenamiento optimizada para Tesla T4/Kaggle
    "batch_size": 1,                            # Batch mínimo para evitar OOM
    "gradient_accumulation": 8,                 # REDUCIDO: Batch efectivo = 8 (era 16)
    "learning_rate": 5e-5,                      # Learning rate para FLAN-T5-Large
    "num_epochs": 1,                            # REDUCIDO: 1 época igual que en modificación anterior
    "warmup_ratio": 0.1,                        # 10% de pasos para warmup
    
    # Configuración de guardado y logging - OPTIMIZADA COMO LLAMA
    "save_steps": 50,                           # IGUAL A LLAMA: Guardar cada 50 pasos
    "eval_steps": 50,                           # IGUAL A LLAMA: Evaluar cada 50 pasos  
    "logging_steps": 5,                         # Log cada 5 pasos
    
    # Directorios
    "output_dir": "./outputs/flan-t5-large-sql",
    "logs_dir": "./logs/flan-t5-large-training",
}
# Configuración LoRA optimizada para FLAN-T5-Large
LORA_CONFIG = {
    "r": 8,                                     # REDUCIDO: Rank 8 en lugar de 16
    "lora_alpha": 16,                           # REDUCIDO: Alpha = 2 * rank
    "target_modules": ["q", "v"],               # REDUCIDO: Solo q y v (no k, o)
    "lora_dropout": 0.1,                        # Dropout para regularización
    "bias": "none",                             # Sin bias para eficiencia
    "task_type": TaskType.SEQ_2_SEQ_LM,         # Tipo de tarea
}

# Configuración de memoria optimizada para FLAN-T5-Large en Tesla T4/Kaggle
MEMORY_CONFIG = {
    # Optimizaciones de modelo
    "torch_dtype": torch.float16,               # FP16 para ahorrar memoria
    "device_map": "auto",                       # Distribución automática
    "load_in_8bit": True,                       # Cuantización 8-bit
    "low_cpu_mem_usage": True,                  # Minimizar uso de RAM
    
    # Optimizaciones de entrenamiento
    "dataloader_pin_memory": False,             # No usar memoria fijada
    "dataloader_num_workers": 0,                # Sin workers paralelos
    "gradient_checkpointing": True,             # Checkpointing activado
    "fp16": True,                               # Entrenar en FP16
}

print("🎯 CONFIGURACIÓN FLAN-T5-LARGE OPTIMIZADA PARA KAGGLE:")
print("=" * 60)
print(f"📦 Modelo: {CONFIG['model_name']}")
print(f"💾 Parámetros: 770M (vs 220M de T5-base)")
print(f"🎮 GPU optimizada: Tesla T4/Kaggle (14.7GB VRAM)")
print(f"📊 Batch efectivo: {CONFIG['batch_size']} × {CONFIG['gradient_accumulation']} = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")
print(f"🔧 LoRA rank: {LORA_CONFIG['r']} (configuración conservadora)")
print(f"⚡ Cuantización: 8-bit ({'✅' if MEMORY_CONFIG['load_in_8bit'] else '❌'})")
print(f"🗺️ Device map: {MEMORY_CONFIG['device_map']}")
print(f"💾 Memoria estimada: 4-6GB (50-60% ahorro)")
print(f"📊 Muestras: {CONFIG['max_samples']} (optimizado para Kaggle)")
print(f"⏰ Épocas: {CONFIG['num_epochs']} (tiempo total ~2-3 horas)")
print(f"💾 Guardado: cada {CONFIG['save_steps']} pasos (prevención OOM)")
print("\n✅ Configuración ultra-conservadora para evitar OutOfMemoryError")

## 4. Preparación de Datos

Preparamos los datos de entrenamiento para text-to-SQL

In [ ]:
def preparar_datos_sql():
    """
    Carga y prepara datos de gretelai/synthetic_text_to_sql para entrenamiento de FLAN-T5-Large
    Usa sql_context (schema), sql_prompt (pregunta) y sql (resultado esperado)
    """
    print(f"📊 Cargando dataset {CONFIG['dataset_name']}...")
    
    # Cargar dataset desde Hugging Face
    dataset = load_dataset(
        CONFIG["dataset_name"], 
        split=CONFIG["dataset_split"]
    )
    
    print(f"✅ Dataset cargado: {len(dataset)} ejemplos totales")
    
    # Convertir a DataFrame para filtrado
    df = dataset.to_pandas()
    
    print(f"📊 Aplicando filtros de calidad...")
    print(f"   Antes del filtrado: {len(df)} ejemplos")
    
    # Filtros de calidad basados en el notebook original
    # 1. SQL válido (no vacío)
    df = df[df['sql'].notna() & (df['sql'].str.len() > 5)]
    print(f"   Después filtro SQL válido: {len(df)}")
    
    # 2. Pregunta válida (mínimo 10 caracteres)
    df = df[df['sql_prompt'].notna() & (df['sql_prompt'].str.len() > 10)]
    print(f"   Después filtro pregunta válida: {len(df)}")
    
    # 3. Contexto/schema válido (mínimo 20 caracteres)
    df = df[df['sql_context'].notna() & (df['sql_context'].str.len() > 20)]
    print(f"   Después filtro contexto válido: {len(df)}")
    
    # 4. Sin errores obvios en el SQL
    df = df[~df['sql'].str.contains('ERROR|error|undefined', case=False, na=False)]
    print(f"   Después filtro errores: {len(df)}")
    
    # 5. Limitar máximo de muestras si se especifica
    if CONFIG["max_samples"] and len(df) > CONFIG["max_samples"]:
        df = df.sample(n=CONFIG["max_samples"], random_state=42).reset_index(drop=True)
        print(f"   Después muestreo aleatorio: {len(df)}")
    
    print(f"\n✅ Dataset final: {len(df)} ejemplos de calidad")
    
    # Mostrar estadísticas del dataset filtrado
    print(f"\n📈 Estadísticas del dataset:")
    print(f"   SQL promedio: {df['sql'].str.len().mean():.0f} caracteres")
    print(f"   Pregunta promedio: {df['sql_prompt'].str.len().mean():.0f} caracteres")
    print(f"   Contexto promedio: {df['sql_context'].str.len().mean():.0f} caracteres")
    
    # Mostrar un ejemplo para verificar la calidad
    print(f"\n📝 Ejemplo del dataset:")
    ejemplo = df.iloc[0]
    print(f"Contexto: {ejemplo['sql_context'][:100]}...")
    print(f"Pregunta: {ejemplo['sql_prompt']}")
    print(f"SQL: {ejemplo['sql']}")
    
    return df

def formatear_para_t5(sql_context, sql_prompt, sql):
    """
    Formatea los datos para FLAN-T5 (modelo instruct)
    Usa formato más natural para aprovechar las capacidades instruct
    """
    # Input: formato natural para modelo instruct
    input_text = f"Schema: {sql_context}\n\nGenerate a SQL query to answer: {sql_prompt}"
    
    # Target: SQL directo sin formato especial
    target_text = sql.strip()
    
    return input_text, target_text

def crear_datasets():
    """
    Crea los datasets de Hugging Face con los datos formateados del dataset gretelai
    """
    print("📦 Creando datasets de Hugging Face...")
    
    # Cargar y filtrar datos
    df_sql = preparar_datos_sql()
    
    # Formatear todos los datos
    inputs = []
    targets = []
    
    for _, row in df_sql.iterrows():
        input_text, target_text = formatear_para_t5(
            row['sql_context'], 
            row['sql_prompt'], 
            row['sql']
        )
        inputs.append(input_text)
        targets.append(target_text)
    
    print(f"✅ {len(inputs)} ejemplos formateados para FLAN-T5")
    
    # División train/eval 80/20
    split_idx = int(len(inputs) * 0.8)
    
    train_inputs = inputs[:split_idx]
    train_targets = targets[:split_idx]
    eval_inputs = inputs[split_idx:]
    eval_targets = targets[split_idx:]
    
    print(f"📈 Dataset entrenamiento: {len(train_inputs)} ejemplos")
    print(f"📊 Dataset evaluación: {len(eval_inputs)} ejemplos")
    
    # Crear datasets de Hugging Face
    train_data = {
        "input_text": train_inputs,
        "target_text": train_targets
    }
    
    eval_data = {
        "input_text": eval_inputs,
        "target_text": eval_targets
    }
    
    train_dataset = HFDataset.from_dict(train_data)
    eval_dataset = HFDataset.from_dict(eval_data)
    
    print(f"✅ Datasets creados")
    print(f"   Train: {len(train_dataset)} ejemplos")
    print(f"   Eval: {len(eval_dataset)} ejemplos")
    
    # Mostrar ejemplo formateado
    print(f"\n📝 Ejemplo de formato:")
    print(f"Input: {train_dataset[0]['input_text'][:100]}...")
    print(f"Target: {train_dataset[0]['target_text']}")
    
    return train_dataset, eval_dataset

# Crear los datasets desde gretelai/synthetic_text_to_sql
train_dataset, eval_dataset = crear_datasets()

print(f"✅ Datos preparados usando {CONFIG['dataset_name']}")
print(f"📊 Formato entrada: 'Schema: [sql_context] Generate a SQL query to answer: [sql_prompt]'")
print(f"🎯 Formato salida: '[sql]'")
print(f"🔄 Campos usados: sql_context (schema), sql_prompt (pregunta), sql (respuesta)")

## 5. Carga del Modelo FLAN-T5-Large

Cargamos el modelo con optimizaciones de memoria para Tesla T4

In [ ]:
def cargar_modelo_t5():
    """
    Carga FLAN-T5-Large con optimizaciones de memoria para Tesla T4
    Incluye device_map='auto' y cuantización de 8-bit
    """
    print(f"🧠 Cargando modelo {CONFIG['model_name']}...")
    print("🔧 Aplicando optimizaciones de memoria avanzadas:")
    print("   • device_map='auto' - distribución automática")
    print("   • load_in_8bit=True - cuantización para ahorrar VRAM") 
    print("   • torch_dtype=torch.float16 - precisión reducida")
    
    # Configurar optimizaciones de memoria
    model_kwargs = {
        "torch_dtype": MEMORY_CONFIG["torch_dtype"],
        "device_map": MEMORY_CONFIG["device_map"],
        "trust_remote_code": True,
        "low_cpu_mem_usage": MEMORY_CONFIG["low_cpu_mem_usage"],
    }
    
    # Agregar cuantización si bitsandbytes está disponible
    try:
        import bitsandbytes as bnb
        model_kwargs.update({
            "load_in_8bit": MEMORY_CONFIG["load_in_8bit"],
            "bnb_8bit_compute_dtype": torch.float16,
            "bnb_8bit_use_double_quant": True,  # Doble cuantización para más eficiencia
        })
        print("   • Cuantización 8-bit: HABILITADA")
    except ImportError:
        print("   • Cuantización 8-bit: NO DISPONIBLE")
        print("   • Para habilitar: pip install bitsandbytes")
        # Remover parámetros de cuantización
        model_kwargs.pop("device_map", None)  # Sin device_map si no hay cuantización
    
    try:
        # Cargar modelo con optimizaciones
        model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            **model_kwargs
        )
        
        # Habilitar gradient checkpointing para ahorrar memoria
        if hasattr(model, 'gradient_checkpointing_enable') and MEMORY_CONFIG["gradient_checkpointing"]:
            model.gradient_checkpointing_enable()
            print("   • Gradient checkpointing: HABILITADO")
        
        print(f"✅ Modelo {CONFIG['model_name']} cargado exitosamente")
        
        # Mostrar información de memoria
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1024**3
                total = torch.cuda.get_device_properties(i).total_memory / 1024**3
                print(f"🎮 GPU {i}: {allocated:.1f}GB / {total:.1f}GB usado")
        
    except Exception as e:
        print(f"❌ Error cargando modelo con optimizaciones: {e}")
        print("🔄 Intentando carga estándar como fallback...")
        
        # Fallback sin optimizaciones avanzadas
        model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            torch_dtype=torch.float16
        )
        print("✅ Modelo cargado con configuración estándar")
    
    # Cargar tokenizer
    print("🔤 Cargando tokenizer...")
    tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"])
    
    # Configurar pad token si no existe
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print("🔧 Pad token configurado")
    
    print(f"✅ Tokenizer cargado para {CONFIG['model_name']}")
    print(f"📊 Vocab size: {len(tokenizer)}")
    
    return model, tokenizer

# Cargar modelo con optimizaciones de memoria
print("🚀 Cargando FLAN-T5-Large con optimizaciones avanzadas...")
base_model, tokenizer = cargar_modelo_t5()
print("✅ Modelo y tokenizer listos con optimizaciones de memoria")

## 6. Tokenización de Datos

Procesamos los datos para convertir texto a tokens que FLAN-T5 pueda entender

In [ ]:
def tokenizar_datos(examples):
    """
    Función para tokenizar los datos para FLAN-T5-Large
    Esta función será aplicada a todo el dataset
    """
    # Tokenizar inputs (pregunta + schema)
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding=False  # No rellenar aquí, lo haremos en el data collator
    )
    
    # Tokenizar targets (SQL)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["target_text"],
            max_length=CONFIG["max_target_length"],
            truncation=True,
            padding=False
        )
    
    # Agregar labels al modelo
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Aplicar tokenización a los datasets
print("🔤 Tokenizando datasets...")

# Tokenizar dataset de entrenamiento
train_dataset = train_dataset.map(
    tokenizar_datos,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Tokenizar dataset de validación
eval_dataset = eval_dataset.map(
    tokenizar_datos,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"✅ Tokenización completada")
print(f"📊 Dataset entrenamiento: {len(train_dataset)} ejemplos tokenizados")
print(f"📊 Dataset validación: {len(eval_dataset)} ejemplos tokenizados")

# Crear data collator para seq2seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
    padding=True,
    return_tensors="pt"
)

print("✅ Data collator configurado para seq2seq")
print(f"📦 Padding dinámico habilitado")
print(f"🔤 Max input length: {CONFIG['max_input_length']}")
print(f"🔤 Max target length: {CONFIG['max_target_length']}")

## 7. Aplicación de LoRA

Aplicamos LoRA (Low-Rank Adaptation) para fine-tuning eficiente

In [ ]:
def aplicar_lora_t5(model):
    """
    Aplica LoRA al modelo FLAN-T5-Large
    Configuración optimizada para 770M parámetros
    """
    print("🔧 Aplicando LoRA a FLAN-T5-Large...")
    
    # Crear configuración LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
    )
    
    print(f"📊 Configuración LoRA:")
    print(f"   Rank (r): {LORA_CONFIG['r']}")
    print(f"   Alpha: {LORA_CONFIG['lora_alpha']}")
    print(f"   Target modules: {LORA_CONFIG['target_modules']}")
    print(f"   Dropout: {LORA_CONFIG['lora_dropout']}")
    print(f"   Task type: {LORA_CONFIG['task_type']}")
    
    # Aplicar LoRA al modelo
    model = get_peft_model(model, lora_config)
    
    # Mostrar información del modelo LoRA
    model.print_trainable_parameters()
    
    print("✅ LoRA aplicado exitosamente")
    print("🎯 Modelo listo para fine-tuning eficiente")
    
    return model

# Aplicar LoRA al modelo base
print("🚀 Aplicando LoRA a FLAN-T5-Large...")
model = aplicar_lora_t5(base_model)

# Verificar memoria después de LoRA
if torch.cuda.is_available():
    print(f"\n💾 Uso de memoria después de LoRA:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"🎮 GPU {i}: {allocated:.1f}GB / {total:.1f}GB usado")

## 8. Configuración de Entrenamiento

Configuramos los argumentos de entrenamiento optimizados para Tesla T4

In [ ]:
# Crear argumentos de entrenamiento optimizados
def crear_training_arguments():
    """Crea argumentos de entrenamiento optimizados para FLAN-T5-Large en Tesla T4"""
    
    # Calcular pasos para warmup
    num_samples = len(train_dataset)
    effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
    steps_per_epoch = num_samples // effective_batch_size
    max_steps = steps_per_epoch * CONFIG["num_epochs"]
    warmup_steps = int(max_steps * CONFIG["warmup_ratio"])
    
    print(f"📊 Configuración de entrenamiento:")
    print(f"   Muestras: {num_samples}")
    print(f"   Batch efectivo: {effective_batch_size}")
    print(f"   Pasos por época: {steps_per_epoch}")
    print(f"   Pasos totales: {max_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    
    training_args = TrainingArguments(
        # Directorios
        output_dir=CONFIG["output_dir"],
        logging_dir=CONFIG["logs_dir"],
        
        # Configuración básica
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        learning_rate=CONFIG["learning_rate"],
        
        # Configuración del scheduler
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        
        # Optimizaciones de memoria - IGUAL A LLAMA
        optim="adamw_8bit",                     # NUEVO: Optimizador eficiente como Llama
        weight_decay=0.01,                      # NUEVO: Weight decay como Llama
        max_grad_norm=1.0,                      # NUEVO: Gradient clipping como Llama
        fp16=torch.cuda.is_available(),         # CAMBIADO: FP16 automático como Llama
        dataloader_pin_memory=MEMORY_CONFIG["dataloader_pin_memory"],
        dataloader_num_workers=MEMORY_CONFIG["dataloader_num_workers"],
        
        # Guardado y logging - OPTIMIZADO COMO LLAMA
        save_steps=CONFIG["save_steps"],
        eval_steps=CONFIG["eval_steps"],
        logging_steps=CONFIG["logging_steps"],
        save_total_limit=2,                     # CAMBIADO: Menos checkpoints como Llama
        
        # Evaluación
        eval_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        
        # Configuración adicional - OPTIMIZADA COMO LLAMA
        dataloader_drop_last=True,              # NUEVO: Como Llama
        remove_unused_columns=False,
        report_to="none",
        seed=42,
        label_names=["labels"],
    )
    
    return training_args

# Crear argumentos de entrenamiento
training_arguments = crear_training_arguments()

# Función de métricas
def compute_metrics(eval_pred):
    """Calcula métricas de evaluación"""
    predictions, labels = eval_pred
    
    # Decodificar predicciones
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Decodificar labels (reemplazar -100 con pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Calcular exactitud
    exact_matches = sum(1 for pred, label in zip(decoded_preds, decoded_labels) 
                       if pred.strip().lower() == label.strip().lower())
    exact_match_rate = exact_matches / len(decoded_preds)
    
    return {"exact_match": exact_match_rate}

# Crear trainer
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("✅ Trainer configurado")
print("🚀 Listo para entrenar FLAN-T5-Large con LoRA")

## 9. Entrenamiento del Modelo

¡Hora de entrenar FLAN-T5-Large para Text-to-SQL!

In [ ]:
# Función para limpiar memoria antes del entrenamiento
def limpiar_memoria():
    """Limpia memoria GPU antes del entrenamiento"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("🧹 Memoria GPU limpiada")

# Limpiar memoria antes de entrenar
limpiar_memoria()

# Mostrar estado inicial de memoria
if torch.cuda.is_available():
    print("💾 Estado de memoria antes del entrenamiento:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        free = total - allocated
        print(f"🎮 GPU {i}: {allocated:.1f}GB usado, {free:.1f}GB libre")

# ENTRENAMIENTO
print("🚀 INICIANDO ENTRENAMIENTO FLAN-T5-LARGE")
print("=" * 50)
print(f"📦 Modelo: {CONFIG['model_name']} (770M parámetros)")
print(f"🎯 Optimizaciones: device_map + 8-bit + LoRA")
print(f"💾 Configuración: {CONFIG['batch_size']} batch × {CONFIG['gradient_accumulation']} acum = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']} efectivo")
print(f"📊 Datos: {len(train_dataset)} entrenamiento + {len(eval_dataset)} validación")
print(f"⏰ Épocas: {CONFIG['num_epochs']}")

try:
    # Ejecutar entrenamiento
    result = trainer.train()
    
    print("\n🎉 ¡ENTRENAMIENTO COMPLETADO EXITOSAMENTE!")
    print(f"📉 Loss final: {result.training_loss:.4f}")
    print(f"⚡ Samples/sec: {result.metrics.get('train_samples_per_second', 0):.2f}")
    
    # GUARDADO OPTIMIZADO PARA EVITAR OOM
    print("\n💾 Guardando modelo de forma optimizada...")
    
    # Limpiar memoria antes de guardar
    limpiar_memoria()
    
    # Guardar solo los adaptadores LoRA (mucho más pequeño)
    print("🔧 Guardando adaptadores LoRA...")
    model.save_pretrained(CONFIG["output_dir"])  # Solo guarda LoRA adapters
    
    # Guardar tokenizer por separado
    print("🔤 Guardando tokenizer...")
    tokenizer.save_pretrained(CONFIG["output_dir"])
    
    # Guardar información del modelo base para reconstrucción
    model_info = {
        "base_model": CONFIG["model_name"],
        "lora_config": LORA_CONFIG,
        "training_config": {
            "learning_rate": CONFIG["learning_rate"],
            "num_epochs": CONFIG["num_epochs"],
            "batch_size": CONFIG["batch_size"],
            "max_samples": CONFIG["max_samples"]
        },
        "final_loss": result.training_loss,
        "training_completed": True
    }
    
    import json
    with open(f"{CONFIG['output_dir']}/model_info.json", "w") as f:
        json.dump(model_info, f, indent=2)
    
    print(f"✅ Modelo LoRA guardado en: {CONFIG['output_dir']}")
    print("📝 Para cargar: usar PeftModel.from_pretrained() sobre el modelo base")
    
    # Evaluación final (si hay memoria)
    try:
        print("\n📊 Realizando evaluación final...")
        eval_results = trainer.evaluate()
        print(f"📊 EVALUACIÓN FINAL:")
        print(f"📉 Eval Loss: {eval_results['eval_loss']:.4f}")
        print(f"🎯 Exact Match: {eval_results['eval_exact_match']:.2%}")
        
        # Guardar resultados de evaluación
        with open(f"{CONFIG['output_dir']}/eval_results.json", "w") as f:
            json.dump(eval_results, f, indent=2)
            
    except Exception as eval_error:
        print(f"⚠️ No se pudo completar evaluación final: {eval_error}")
        print("💡 Pero el modelo se guardó correctamente")
    
except Exception as e:
    print(f"\n❌ Error durante el entrenamiento: {e}")
    print("💡 Sugerencias:")
    print("   • Verificar memoria GPU disponible")
    print("   • Reducir batch_size si es necesario")
    print("   • Instalar bitsandbytes para optimizaciones")
    
finally:
    # Limpiar memoria al final (MUY IMPORTANTE)
    print("\n🧹 Limpieza final de memoria...")
    
    # Liberar referencias del modelo
    del model
    del base_model
    if 'trainer' in locals():
        del trainer
    
    # Limpiar memoria agresivamente
    limpiar_memoria()
    
    # Forzar garbage collection múltiple
    for _ in range(3):
        gc.collect()
    
    print("✅ Memoria liberada")

print("\n✅ Notebook completado - FLAN-T5-Large entrenado para Text-to-SQL")
print("💡 IMPORTANTE: Solo se guardaron los adaptadores LoRA, no el modelo completo")
print("🔄 Para usar: cargar modelo base + adaptadores LoRA")

## 10. Prueba del Modelo Entrenado (Opcional)

Probamos el modelo con algunos ejemplos para verificar su funcionamiento

## 11. Carga del Modelo LoRA Entrenado

Cómo cargar y usar el modelo LoRA entrenado (para evaluación posterior)

In [ ]:
def cargar_modelo_lora_entrenado(model_path="./outputs/flan-t5-large-sql"):
    """
    Carga el modelo LoRA entrenado para evaluación
    """
    print("🔄 CARGANDO MODELO LORA ENTRENADO")
    print("=" * 40)
    
    try:
        # Verificar que existen los archivos
        import os
        if not os.path.exists(model_path):
            print(f"❌ No se encontró el directorio: {model_path}")
            return None, None
        
        # Cargar información del modelo
        info_path = f"{model_path}/model_info.json"
        if os.path.exists(info_path):
            with open(info_path, "r") as f:
                model_info = json.load(f)
            print(f"📋 Información del modelo encontrada:")
            print(f"   Modelo base: {model_info['base_model']}")
            print(f"   Loss final: {model_info.get('final_loss', 'N/A')}")
            print(f"   Épocas: {model_info['training_config']['num_epochs']}")
        
        # Cargar modelo base
        print("🧠 Cargando modelo base...")
        base_model = T5ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"],
            torch_dtype=torch.float16,
            device_map="auto",
            load_in_8bit=True
        )
        
        # Cargar adaptadores LoRA
        print("🔧 Cargando adaptadores LoRA...")
        from peft import PeftModel
        lora_model = PeftModel.from_pretrained(base_model, model_path)
        
        # Cargar tokenizer
        print("🔤 Cargando tokenizer...")
        trained_tokenizer = T5Tokenizer.from_pretrained(model_path)
        
        print("✅ Modelo LoRA cargado exitosamente")
        print("🎯 Listo para inferencia y evaluación")
        
        return lora_model, trained_tokenizer
        
    except Exception as e:
        print(f"❌ Error cargando modelo LoRA: {e}")
        print("💡 Asegúrate de que el entrenamiento se completó correctamente")
        return None, None

# Ejemplo de carga (descomenta para usar)
# trained_model, trained_tokenizer = cargar_modelo_lora_entrenado()

print("📝 Para cargar el modelo entrenado:")
print("   1. Descomenta la línea de carga arriba")
print("   2. Asegúrate de que existe el directorio de salida")
print("   3. Usa trained_model y trained_tokenizer para inferencia")

print("\n🔄 GUÍA DE RECUPERACIÓN SI NO SE GUARDÓ:")
print("1. Ejecuta solo las celdas de preparación (1-8)")
print("2. Salta el entrenamiento y carga un modelo pre-existente")
print("3. O re-entrena con configuración optimizada de guardado")

In [ ]:
def probar_modelo_entrenado():
    """Prueba el modelo entrenado con ejemplos del dataset"""
    print("🧪 PROBANDO MODELO FLAN-T5-LARGE ENTRENADO")
    print("=" * 50)
    
    # Ejemplos de prueba del formato original
    test_cases = [
        {
            "sql_context": "CREATE TABLE users (id INT, name VARCHAR(50), age INT, city VARCHAR(50));",
            "sql_prompt": "Get all users older than 25"
        },
        {
            "sql_context": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL, category VARCHAR(50));",
            "sql_prompt": "Find the most expensive product"
        },
        {
            "sql_context": "CREATE TABLE employees (id INT, name VARCHAR(50), department VARCHAR(50), salary DECIMAL);",
            "sql_prompt": "What is the average salary by department?"
        }
    ]
    
    for i, test in enumerate(test_cases, 1):
        print(f"\n🧪 PRUEBA {i}:")
        print(f"📋 Schema: {test['sql_context'][:80]}...")
        print(f"❓ Pregunta: {test['sql_prompt']}")
        
        try:
            # Formatear input usando la misma función del entrenamiento
            input_text, _ = formatear_para_t5(
                test['sql_context'], 
                test['sql_prompt'], 
                ""  # SQL vacío para la prueba
            )
            
            # Tokenizar
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=CONFIG["max_input_length"]
            )
            
            # Mover al device del modelo
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            
            # Generar SQL
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_length=CONFIG["max_target_length"],
                    num_beams=3,
                    early_stopping=True,
                    do_sample=False
                )
            
            # Decodificar resultado
            generated_sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
            print(f"✅ SQL generado: {generated_sql}")
            
        except Exception as e:
            print(f"❌ Error en generación: {e}")
        
        print("-" * 60)

# Ejecutar pruebas (solo si el entrenamiento fue exitoso)
# Descomenta la siguiente línea para probar el modelo
# probar_modelo_entrenado()

print("📝 Para probar el modelo, descomenta la llamada a probar_modelo_entrenado()")
print("🔄 Usa el mismo formato: sql_context, sql_prompt como en el dataset gretelai")
print("✅ Compatible con el formato del notebook train_t5_sql.ipynb original")